In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

df = pd.read_csv("../../data/raw/2020-Apr-L.csv")

In [2]:
# 1. AGREGACIÓN: eventos totales por user_id
# ─────────────────────────────────────────────
user_counts = df.groupby("user_id").size()
 
total_usuarios = len(user_counts)
media          = user_counts.mean()
mediana        = user_counts.median()
p90            = np.percentile(user_counts, 90)
pct_cold_start = (user_counts <= 3).sum() / total_usuarios * 100
 
print("=== MÉTRICAS OBLIGATORIAS ===")
print(f"Total usuarios  : {total_usuarios:,}")
print(f"Media           : {media:.1f} interacciones")
print(f"Mediana         : {mediana:.0f} interacciones")
print(f"P90             : {p90:.0f} interacciones")
print(f"Usuarios ≤3     : {pct_cold_start:.1f}%")

=== MÉTRICAS OBLIGATORIAS ===
Total usuarios  : 1,379,296
Media           : 7.0 interacciones
Mediana         : 3 interacciones
P90             : 16 interacciones
Usuarios ≤3     : 59.2%


In [3]:
# HISTOGRAMA EN ESCALA LOG-LOG
log_min = 0
log_max = np.ceil(np.log10(user_counts.max()))
bins    = np.logspace(log_min, log_max, num=50)
 
counts, edges = np.histogram(user_counts, bins=bins)
 
# Centros de bin (media geométrica)
bin_centers = np.sqrt(edges[:-1] * edges[1:])
 
# Filtrar bins vacíos
mask    = counts > 0
x_hist  = bin_centers[mask]
y_hist  = counts[mask]
 

In [4]:
# 3. REGRESIÓN LINEAL SOBRE LA COLA
#    Solo bins donde x > mediana (cola derecha)
# ─────────────────────────────────────────────
cola_mask = x_hist > mediana
x_cola    = np.log10(x_hist[cola_mask])
y_cola    = np.log10(y_hist[cola_mask])
 
slope, intercept, r_value, p_value, _ = stats.linregress(x_cola, y_cola)
r2 = r_value ** 2
 
x_fit_log = np.linspace(x_cola.min(), x_cola.max(), 100)
y_fit_log = slope * x_fit_log + intercept
x_fit     = 10 ** x_fit_log
y_fit     = 10 ** y_fit_log
 
print(f"\n=== REGRESIÓN (cola) ===")
print(f"Pendiente (α)   : {-slope:.3f}")
print(f"R²              : {r2:.4f}")


=== REGRESIÓN (cola) ===
Pendiente (α)   : 1.662
R²              : 0.9482


In [6]:
# 4. FIGURA
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor("#F8FAFC")
ax.set_facecolor("#F8FAFC")
 
# ── Barras del histograma ─────────────────────
ax.bar(x_hist, y_hist,
       width=np.diff(np.concatenate([[x_hist[0]*0.5], x_hist])) * 0.85,
       color="#60A5FA", alpha=0.75, edgecolor="white",
       linewidth=0.5, zorder=3, label="Usuarios por nivel de actividad")
 
# ── Línea de regresión (cola) ─────────────────
ax.plot(x_fit, y_fit,
        color="#DC2626", linewidth=2, linestyle="--", zorder=5,
        label=f"Ajuste power law (cola): α = {-slope:.2f},  R² = {r2:.3f}")
 
# ── Líneas verticales de métricas ────────────
for val, label, color, pos in [
    (mediana, f"Mediana\n{int(mediana):,}", "#059669", "right"),
    (media,   f"Media\n{int(media):.0f}",  "#D97706", "right"),
    (p90,     f"P90\n{int(p90):,}",        "#7C3AED", "left"),
]:
    ax.axvline(val, color=color, linewidth=1.3, linestyle=":", zorder=4)
    ax.text(
        val * (0.88 if pos == "right" else 1.08),
        y_hist.max() * 0.6,
        label,
        fontsize=8, color=color, fontweight="bold",
        va="center", ha=pos,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=color, lw=0.8, alpha=0.85)
    )
 
# ── Anotación Cold Start ──────────────────────
ax.text(0.02, 0.08,
        f"{pct_cold_start:.1f}% de usuarios\ntienen ≤ 3 interacciones\n(Cold Start — Fase 4.3)",
        transform=ax.transAxes,
        fontsize=8.5, color="#1E293B",
        va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="#FEF9C3", ec="#FDE047", lw=1))
 
# ── Escala log-log ────────────────────────────
ax.set_xscale("log")
ax.set_yscale("log")
 
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x):,}" if x >= 1 else ""))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{int(x):,}" if x >= 1 else ""))
 
ax.tick_params(axis="both", which="both", length=0, labelsize=9)
 
# Cuadrícula suave
ax.grid(True, which="major", color="#E2E8F0", linewidth=0.8, zorder=1)
ax.grid(True, which="minor", color="#F1F5F9", linewidth=0.4, zorder=1)
 
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#E2E8F0")
ax.spines["bottom"].set_color("#E2E8F0")
 
# ── Ejes ─────────────────────────────────────
ax.set_xlabel("Número de Interacciones por Usuario (escala log)", fontsize=10,
              color="#334155", labelpad=10)
ax.set_ylabel("Número de Usuarios (escala log)", fontsize=10,
              color="#334155", labelpad=10)
 
# ── Títulos ───────────────────────────────────
ax.set_title("Distribución de Interacciones por Usuario (Ley de Potencias)",
             fontsize=13, fontweight="bold", loc="left",
             pad=28, color="#0F172A")
ax.text(0.0, 1.045,
        "Histograma log-log · Unidad observacional: user_id · Datos: 2020-04-01 → 2020-04-30",
        transform=ax.transAxes, fontsize=8.5, color="#64748B")
 
# ── Leyenda ───────────────────────────────────
ax.legend(fontsize=8.5, frameon=True, loc="upper right",
          framealpha=0.9, edgecolor="#CBD5E1")
 
# ── Estadísticas resumen ──────────────────────
stats_text = (
    f"n usuarios = {total_usuarios:,}\n"
    f"Media      = {media:.1f}\n"
    f"Mediana    = {int(mediana):,}\n"
    f"P90        = {int(p90):,}"
)
ax.text(0.99, 0.35, stats_text,
        transform=ax.transAxes,
        fontsize=8, color="#475569", family="monospace",
        va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", fc="#F1F5F9", ec="#CBD5E1", lw=1))
 
plt.tight_layout()
plt.savefig("outputs/05_dist_user.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print("¡Gráfico 05_dist_user.png guardado exitosamente!")

¡Gráfico 05_dist_user.png guardado exitosamente!
